>**Projeto Processamento de Big Data** 


>**Grupo 4**: Camila Sousa 111017 | Carolina Brunheta 110888 | Miguel Correia 110786


>**2023/24**

>**Treino e Ajuste do Modelo**

# Inicializar sessão

In [1]:
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import abs, col
from pyspark.ml.regression import GBTRegressor
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("LondonBikeShareUsage") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.maxResultSize", "2048m") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memoryOverhead", "4g") \
    .getOrCreate()


In [2]:
spark

# Importação do ficheiro Parquet

In [3]:
filename = ["../notebooks/london_small.parquet"]
london_small = spark.read.format("parquet").load(filename)

In [4]:
# Filtrar observações com duração igual a 0.00
london_small = london_small.filter(col("duration") > 60)

In [5]:
print(london_small.count())

11099137


In [6]:
print("Número de linhas: ", london_small.count(), "      ||   Número de colunas: ", len(london_small.columns))

Número de linhas:  11099140       ||   Número de colunas:  10


In [6]:
london_small.printSchema()

root
 |-- duration: float (nullable = true)
 |-- end_station_id: float (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- dayofweek: integer (nullable = true)
 |-- start_hour: integer (nullable = true)
 |-- start_period_of_day: string (nullable = true)
 |-- season: string (nullable = true)
 |-- distance_between_stations: float (nullable = true)
 |-- closeness_centrality_start: float (nullable = true)
 |-- eigenvector_centrality_end: float (nullable = true)



# Aprendizagem Supervisionada

- Prever 'duration'

Vamos utilizar o conjunto de dados menor para estas fases por questões de otimização de tempo e recursos

### Divisião do dataset em conjunto treino, teste

In [7]:
london_train, london_validation, london_test = london_small.randomSplit([0.7, 0.15, 0.15], 123)
print(f"Existem {london_train.count()} linhas no conjunto de treino, {london_validation.count()} linhas no conjunto de validação e {london_test.count()} no conjunto de teste.")

Existem 7767952 linhas no conjunto de treino, 1665389 linhas no conjunto de validação e 1665796 no conjunto de teste.


##### Dummies:

O próximo passo é codificar as colunas categóricas em índices numéricos, que podem ser usados como input para o modelo. Para isso, é usado um objeto designado StringIndexer. Em seguida, é criado um objeto OneHotEncoder que transforma os índices numéricos em vetores binários, que são mais adequados para o modelo.

Por fim, é usado um objeto VectorAssembler para juntar todas as colunas num único vetor, que será o input do modelo de classificação. As colunas de entrada são especificadas como assembler_inputs, que contém os nomes das colunas categóricas e não categóricas codificadas e os seus índices numéricos correspondentes. O vetor resultante é atribuído à coluna features.


In [8]:
# Encoding columns and vector assembling them
num_cols = ['end_station_id', 'start_station_id','start_hour', 'distance_between_stations' ]

cat_cols = ['season', 'start_period_of_day', 'dayofweek']

index_output_cols = [x + ' Index' for x in cat_cols] #Lista das colunas resultantes da indexação das variáveis categóricas
ohe_output_cols = [x + ' OHE' for x in cat_cols] # Lista das colunas resultantes da codificação one-hot das variáveis categóricas indexadas

#Transformar as colunas categóricas em colunas de índices numéricos, para posteriormente aplicar a codificação one-hot.
string_indexer = StringIndexer(inputCols=cat_cols, outputCols=index_output_cols, handleInvalid="skip")

#Transformar as colunas de índices numéricos em colunas de codificação one-hot. 
ohe_encoder = OneHotEncoder(inputCols=index_output_cols, outputCols=ohe_output_cols) #Cada valor único numa coluna categórica torna-se uma nova coluna binária
   
#Combinar as colunas de codificação one-hot e as colunas numéricas numa lista única (vetor)
assembler_inputs = ohe_output_cols + num_cols
vec_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
assembler_inputs

['season OHE',
 'start_period_of_day OHE',
 'dayofweek OHE',
 'end_station_id',
 'start_station_id',
 'start_hour',
 'distance_between_stations']

##### Normalização:

Dado o comportamento inerente ao algoritmo Random Forest Regressor, baseado em árvores de decisão, que pretendemos estudar, optamos por não realizar a normalização dos dados neste projeto. Esta decisão garante a interpretabilidade dos resultados sem comprometer o desempenho preditivo do modelo. Esta normalização deve ser feita caso sejam estudados outros algoritmos com SVM, ou para uma abordagem de aprensizagem não supervizionada, usando Knn ou PCA por exemplo.

# Algoritmo Random Forest


In [9]:
regressor = RandomForestRegressor(featuresCol='features', labelCol= 'duration')

### Definir ML Pipeline

Encapsular estas etapas em um pipeline usando Pipeline do scikit-learn:
- vec_assembler
- ML estimator

In [10]:
pipeline = Pipeline(stages=[string_indexer, ohe_encoder, vec_assembler, regressor])

### Ajustar o pipeline aos dados de treinamento usando o método fit

Criou-se um modelo pipeline com os dados do conjunto de treino

In [11]:
pipeline_model = pipeline.fit(london_train)

###  Avalie o desempenho do modelo utilizando métricas relevantes.

In [12]:
# Fazer previsões no conjunto de teste 
london_prediction = pipeline_model.transform(london_validation)

london_prediction.printSchema()

root
 |-- duration: float (nullable = true)
 |-- end_station_id: float (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- dayofweek: integer (nullable = true)
 |-- start_hour: integer (nullable = true)
 |-- start_period_of_day: string (nullable = true)
 |-- season: string (nullable = true)
 |-- distance_between_stations: float (nullable = true)
 |-- closeness_centrality_start: float (nullable = true)
 |-- eigenvector_centrality_end: float (nullable = true)
 |-- season Index: double (nullable = false)
 |-- start_period_of_day Index: double (nullable = false)
 |-- dayofweek Index: double (nullable = false)
 |-- season OHE: vector (nullable = true)
 |-- start_period_of_day OHE: vector (nullable = true)
 |-- dayofweek OHE: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- prediction: double (nullable = false)



Este output contém as seguintes colunas do DataFrame london_prediction:

- 'duration': variável alvo que indica o tempo de duração de uma viagem de bicicleta.

- 'features': variáveis preditoras em formato vetor que passam pelo processo de codificação e vetorização.

- 'prediction': previsão final do modelo para a variável alvo 'duration'


### Avaliar o Modelo

In [14]:
# Calcular RMSE
evaluator_rmse = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='rmse')
rmse_rf1 = evaluator_rmse.evaluate(london_prediction)

# Calcular R-squared
evaluator_r2 = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='r2')
r2_rf1 = evaluator_r2.evaluate(london_prediction)

# Calcular MAPE manualmente
mape_rf1 = london_prediction.withColumn("abs_error", abs(col("duration") - col("prediction")) / col("duration")) \
                        .selectExpr("avg(abs_error) as MAPE") \
                        .collect()[0]["MAPE"]

metrics_data = [
    ("RMSE", rmse_rf1),
    ("R2", r2_rf1),
    ("MAPE", mape_rf1)
]
# Crie um DataFrame a partir da lista de tuplas
metrics_df = spark.createDataFrame(metrics_data, ["Metric", "Value"])
metrics_df.show()

+------+------------------+
|Metric|             Value|
+------+------------------+
|  RMSE|  613.903735466818|
|    R2| 0.308908685064913|
|  MAPE|0.6146910876670256|
+------+------------------+



In [15]:
metrics_df.write.mode("overwrite").csv("./metrics1.csv", header=True)

### Tune Model: Ajustar o modelo

Para melhorar o modelo podemos considerar ajustar os parâmetros do algoritmo. 

- n_estimators: O número de árvores na floresta.
- max_depth: A profundidade máxima de cada árvore na floresta.
- min_samples_split: O número mínimo de amostras necessárias para dividir um nó interno.
- min_samples_leaf: O número mínimo de amostras necessárias para ser um nó folha.
- max_features: O número máximo de características a serem consideradas ao procurar a melhor divisão.
- bootstrap: Se amostras são desenhadas com substituição (True) ou não (False).
- random_state: A semente usada pelo gerador de números aleatórios.
- n_jobs: O número de trabalhos em paralelo a serem executados durante o ajuste e a previsão (-1 para usar todos os processadores disponíveis).

Para isto podemos considerar usar GridSearch. Para melhorar o modelo podemos fazer também validação cruzada.

In [16]:
# Definir o regressor RandomForestRegressor
regressor = RandomForestRegressor(featuresCol='features', labelCol= 'duration',numTrees=50, maxDepth=5, seed=42)
# Criar pipeline
pipeline2 = Pipeline(stages=[string_indexer, ohe_encoder, vec_assembler, regressor])
# Ajustar o modelo do pipeline aos dados de treinamento
pipeline_model2 = pipeline2.fit(london_train)
# previsões nos dados de teste
london_prediction2 = pipeline_model2.transform(london_validation)

# Avaliar o modelo com RMSE
evaluator_rmse = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='rmse')
rmse_rf2 = evaluator_rmse.evaluate(london_prediction2)

# Avaliar o modelo com R-squared
evaluator_r2 = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='r2')
r2_rf2 = evaluator_r2.evaluate(london_prediction2)

# Calcule MAPE manualmente
mape_rf2 = london_prediction2.withColumn("abs_error", abs(col("duration") - col("prediction")) / col("duration")) \
                        .selectExpr("avg(abs_error) as MAPE") \
                        .collect()[0]["MAPE"]

metrics_data2 = [
    ("RMSE", rmse_rf2),
    ("R2", r2_rf2),
    ("MAPE", mape_rf2)
]
metrics_df2 = spark.createDataFrame(metrics_data2, ["Metric", "Value"])
metrics_df2.show()

+------+------------------+
|Metric|             Value|
+------+------------------+
|  RMSE| 615.3242566450768|
|    R2|0.3057067314397435|
|  MAPE|0.6172106073684386|
+------+------------------+



In [17]:
# Definir o regressor RandomForestRegressor
regressor = RandomForestRegressor(featuresCol='features', labelCol= 'duration',numTrees=50, maxDepth=10, seed=42)
# Criar pipeline
pipeline3 = Pipeline(stages=[string_indexer, ohe_encoder, vec_assembler, regressor])
# Ajustar o modelo do pipeline aos dados de treinamento
pipeline_model3 = pipeline3.fit(london_train)
# Fazer as previsões nos dados de teste
london_prediction3 = pipeline_model3.transform(london_validation)

# Avaliar o modelo com RMSE
evaluator_rmse = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='rmse')
rmse_rf3 = evaluator_rmse.evaluate(london_prediction3)

# Avaliar o modelo com R-squared
evaluator_r2 = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='r2')
r2_rf3 = evaluator_r2.evaluate(london_prediction3)

# Calcular MAPE manualmente
mape_rf3 = london_prediction3.withColumn("abs_error", abs(col("duration") - col("prediction")) / col("duration")) \
                        .selectExpr("avg(abs_error) as MAPE") \
                        .collect()[0]["MAPE"]

metrics_data3 = [
    ("RMSE", rmse_rf3),
    ("R2", r2_rf3),
    ("MAPE", mape_rf3)
]
metrics_df3 = spark.createDataFrame(metrics_data3, ["Metric", "Value"])
metrics_df3.show()

+------+-------------------+
|Metric|              Value|
+------+-------------------+
|  RMSE|  589.9946671306155|
|    R2|0.36169087130107247|
|  MAPE| 0.5255280355987231|
+------+-------------------+



In [18]:
metrics_df3.write.mode("overwrite").csv("./metrics3.csv", header=True)

In [19]:
# Definir o regressor RandomForestRegressor
regressor = RandomForestRegressor(featuresCol='features', labelCol= 'duration',numTrees=100, maxDepth=5, seed=42)
# Criar pipeline
pipeline4 = Pipeline(stages=[string_indexer, ohe_encoder, vec_assembler, regressor])
# Ajustar o modelo do pipeline aos dados de treinamento
pipeline_model4 = pipeline4.fit(london_train)
# Fazer as previsões nos dados de teste
london_prediction4 = pipeline_model4.transform(london_validation)

# Avaliar o modelo com RMSE
evaluator_rmse = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='rmse')
rmse_rf4 = evaluator_rmse.evaluate(london_prediction4)

# Avaliar o modelo com R-squared
evaluator_r2 = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='r2')
r2_rf4 = evaluator_r2.evaluate(london_prediction4)

# Calcular MAPE manualmente
mape_rf4 = london_prediction4.withColumn("abs_error", abs(col("duration") - col("prediction")) / col("duration")) \
                        .selectExpr("avg(abs_error) as MAPE") \
                        .collect()[0]["MAPE"]

metrics_data4 = [
    ("RMSE", rmse_rf4),
    ("R2", r2_rf4),
    ("MAPE", mape_rf4)
]
metrics_df4 = spark.createDataFrame(metrics_data4, ["Metric", "Value"])
metrics_df4.show()

+------+------------------+
|Metric|             Value|
+------+------------------+
|  RMSE| 616.5400141147315|
|    R2|0.3029604522322037|
|  MAPE| 0.622223145748042|
+------+------------------+



# Gradient-Boosted Trees Regressor

Embora o GBTRegressor não tenha sido abordado em aula, consideramos interessante experimentá-lo devido às suas diferenças em relação ao RandomForestRegressor. Enquanto o RandomForestRegressor constrói múltiplas árvores de decisão independentes e as combina para fazer previsões, o GBTRegressor utiliza um processo de boosting sequencial, onde cada árvore é ajustada aos erros residuais do modelo anterior. Isso pode resultar num modelo mais poderoso, capaz de capturar relações mais complexas entre as características e a variável de destino. Portanto, decidimos explorar o GBTRegressor para avaliar o seu desempenho e capacidade de capturar padrões nos nossos dados.

In [21]:
GBTR_regressor = GBTRegressor(maxIter=20, featuresCol='features', labelCol='duration')
### Definir ML Pipeline

pipeline_GBTR = Pipeline(stages=[string_indexer, ohe_encoder, vec_assembler, GBTR_regressor])
### Ajustar o pipeline aos dados de treinamento usando o método fit
pipeline_model_GBTR = pipeline_GBTR.fit(london_train)
###  Avaliar o desempenho do modelo utilizando métricas relevantes.
# Fazer previsões no conjunto de teste 
london_prediction_GBTR = pipeline_model_GBTR.transform(london_validation)
london_prediction_GBTR.printSchema()
london_prediction_GBTR.select('features', 'prediction', 'duration').show(truncate=False)
### Avaliar o Modelo

# Calcular RMSE
evaluator_rmse_GBTR = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='rmse')
rmse_GBTR = evaluator_rmse_GBTR.evaluate(london_prediction_GBTR)
print(f"Erro Quadrático Médio (RMSE) nos dados de teste = {rmse_GBTR}")

# Calcular MAPE manualmente
mape_GBTR = london_prediction_GBTR.withColumn("abs_error", abs(col("duration") - col("prediction")) / col("duration")) \
                        .selectExpr("avg(abs_error) as MAPE") \
                        .collect()[0]["MAPE"]
print(f"Erro Percentual Absoluto Médio (MAPE) nos dados de teste = {mape_GBTR}")

# Calcular R-squared
evaluator_r2_GBTR = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='r2')
r2_GBTR = evaluator_r2_GBTR.evaluate(london_prediction_GBTR)
print(f"R-Quadrado (R2) nos dados de teste = {r2_GBTR}")

root
 |-- duration: float (nullable = true)
 |-- end_station_id: float (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- dayofweek: integer (nullable = true)
 |-- start_hour: integer (nullable = true)
 |-- start_period_of_day: string (nullable = true)
 |-- season: string (nullable = true)
 |-- distance_between_stations: float (nullable = true)
 |-- closeness_centrality_start: float (nullable = true)
 |-- eigenvector_centrality_end: float (nullable = true)
 |-- season Index: double (nullable = false)
 |-- start_period_of_day Index: double (nullable = false)
 |-- dayofweek Index: double (nullable = false)
 |-- season OHE: vector (nullable = true)
 |-- start_period_of_day OHE: vector (nullable = true)
 |-- dayofweek OHE: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- prediction: double (nullable = false)

+--------------------------------------------------------------------------+------------------+--------+
|features                           

Contudo os resultados não mostraram melhorias significativas em comparação com o melhor modelo até então. Nota-se que é ligeiramente melhor, talvez ao iterar mais vezes melhorasse o desempenho do modelo, contudo não nos é viável a nível de recursos, considerar um conjunto de validação menor que o de teste para afinar o modelo seria o mais adequado. 
Visto que o GradientBoosted Tree Regressor é mais complexo que uma random forest, e que os resultados são os mesmos, considerámos a melhor Random Forest







 

# Melhor modelo

O melhor modelo obtido foi uma Random Forest com 50 árvores e profundidade máxima de 10

In [22]:


# Valores das métricas para cada modelo
metrics_values = {
    "RF1": {"RMSE": rmse_rf1, "R2": r2_rf1, "MAPE": mape_rf1},
    "RF2": {"RMSE": rmse_rf2, "R2": r2_rf2, "MAPE": mape_rf2},
    "RF3": {"RMSE": rmse_rf3, "R2": r2_rf3, "MAPE": mape_rf3},
    "RF4": {"RMSE": rmse_rf4, "R2": r2_rf4, "MAPE": mape_rf4},
    "GBTR": {"RMSE": rmse_GBTR, "R2": r2_GBTR, "MAPE": mape_GBTR}
}

# Criar DataFrame
df = pd.DataFrame(metrics_values).T

# Imprimir como tabela
print(df)

            RMSE        R2      MAPE
RF1   613.903735  0.308909  0.614691
RF2   615.324257  0.305707  0.617211
RF3   589.994667  0.361691  0.525528
RF4   616.540014  0.302960  0.622223
GBTR  586.365291  0.369520  0.492955


Guardar em disco:

In [ ]:
pipeline3.save("./notebooks/model-pipeline-RandomForestRegression")

In [24]:
pipeline_model3.save("./notebooks/model-RandomForestRegression")

In [25]:
output_london_test = "london_test.parquet"
london_test.write.mode("overwrite").parquet(output_london_test)

In [26]:
output_london_test = "./notebooks/london_test.parquet"
london_test.write.mode("overwrite").parquet(output_london_test)

In [27]:
ls -la

total 2316
drwxrwxrwx 1 root   root     4096 Jun 11 05:49  ./
drwxrwxrwx 1 root   root     4096 Jun 10 09:06  ../
-rwxrwxrwx 1 root   root   161916 Jun 11 04:20 'A (1).ipynb'*
-rwxrwxrwx 1 root   root  1771987 Jun 11 05:07  Analise_exploratoria_dos_dados.ipynb*
-rwxrwxrwx 1 root   root     5458 Jun 11 04:20  Aplicacao_do_Modelo.ipynb*
-rw-r--r-- 1 jovyan users   28208 Jun 10 19:54  duracao_media_alugueres_dias_uteis.png
-rw-r--r-- 1 jovyan users   24705 Jun 10 19:54  duracao_media_alugueres_fim_semana.png
-rw-r--r-- 1 jovyan users   40163 Jun 10 19:44  duracao_media_alugueres_por_mes.png
-rw-r--r-- 1 jovyan users  111062 Jun 11 05:01  Importacao_e_depuracao_dos_dados.ipynb
drwxr-xr-x 1 jovyan users    4096 Jun 11 04:01  london_count_final.csv/
drwxr-xr-x 1 jovyan users    4096 Jun 10 20:36  london_counts.csv/
drwxr-xr-x 1 jovyan users    4096 Jun 10 20:41  london_nulls.csv/
drwxr-xr-x 1 jovyan users    4096 Jun 11 05:07  london.parquet/
drwxr-xr-x 1 jovyan users    4096 Jun 11 05:06  l